In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path

def load_environment_variables():
    """
    Loads the environment variables from .env.
    Loads OpenAI api key and target model name used in the experiment
    New environment variables as necessary will be appended
    """

    PROJECT_ROOT = Path.cwd().parent
    ENV_FILE = PROJECT_ROOT / ".env"

    if ENV_FILE.exists():
        print(
            f".env file located at: {ENV_FILE.resolve()}")
        
        load_dotenv()
        env = {
                "api_key":              os.getenv("OPENAI_API_KEY"),
                "model_name":           os.getenv("OPENAI_MODEL_NAME"),
                "input_token_price":    float(os.getenv('MODEL_INPUT_TOKEN_PRICE')),
                "output_token_price":   float(os.getenv('MODEL_OUTPUT_TOKEN_PRICE'))
            }

    else:
        print(
            f".env not found at: {ENV_FILE.resolve()}"
        )
        env = {
            "api_key":              None,
            "model_name":           None,
            "input_token_price":    None,
            "output_token_price":   None
        }

    return env

In [2]:
from openai import OpenAI

env = load_environment_variables()

client = OpenAI(
    api_key=env['api_key']
)

response = client.chat.completions.create(
    model=env['model_name'],
    messages=[
        {
            "role": "user",
            "content": "what is RAG?"
        }
    ]
)

print(response.choices[0].message.content)

.env file located at: D:\Bits-lab\eduassist-llmops-prompt-evaluation\.env
RAG can refer to different concepts depending on the context. Here are a few possibilities:

1. **Red, Amber, Green (RAG) Status**: This is often used in project management and reporting to indicate the status or progress of a project or task. 
   - **Red** indicates significant issues, delays, or problems.
   - **Amber** signifies that there are some concerns but that they are manageable.
   - **Green** shows that everything is on track and proceeding as planned.

2. **RAG Framework in AI**: In the context of artificial intelligence, particularly in machine learning and natural language processing, RAG refers to "Retrieval-Augmented Generation." This is a technique that combines retrieval-based methods with generative models to improve the quality and relevance of generated text. It allows the model to pull in external information during the generation process, making responses more accurate and contextually rel

In [3]:
response = client.chat.completions.create(
    model=env['model_name'],
    messages=[
        {
            "role": "user",
            "content": "what is RAG in Natural Language Processing?"
        }
    ]
)

print(response.choices[0].message.content)

RAG, or Retrieval-Augmented Generation, is a framework in Natural Language Processing that combines the capabilities of retrieval-based methods with generative models. The main idea is to enhance the generative capabilities of language models by providing them with relevant external information retrieved from a knowledge base or a document corpus. Here's a breakdown of how RAG works:

1. **Retrieval Component**: Given a query or input, the RAG system retrieves relevant documents or pieces of information from a large corpus. This step usually uses an information retrieval technique, such as TF-IDF, BM25, or neural retrievers based on embedding methods.

2. **Augmentation with Retrieved Information**: The relevant information obtained from the retrieval step is then combined with the initial input. This additional context helps to inform and guide the generative process.

3. **Generation Component**: A generative language model, such as a transformer-based model (e.g., BART, T5), takes t

In [4]:
questions = [
    "what is overfitting in machine learning?",
    "Explain gradient descent in simple terms.",
    "what is neural network?",
    "Difference between supervised and unsupervised learning?",
    "Explain backpropagation in neural networks."
]

In [5]:
PROMT_V1 = """
Answer the following student questions clearly.
Question: {Question}
"""

In [6]:
import time
import pandas as pd

def evaluate_prompt(
        questions,
        prompt_template,
        prompt_version,
        model,
        input_token_price,
        output_token_price
):
    results = []

    for idx, question in enumerate(questions, start=1):

        prompt = prompt_template.format(
            Question=question
        )
        
        start_time = time.time()

        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        end_time = time.time()

        latency = end_time - start_time

        # token cost calculation
        input_token_price = env['input_token_price']
        output_token_price = env['output_token_price']

        input_cost = (
            response.usage.prompt_tokens / (1024 * 1024) 
        ) * input_token_price

        output_cost = (
            response.usage.completion_tokens / (1024 * 1024)
        ) * output_token_price

        total_cost = input_cost + output_cost
        
        results.append(
            {
                "prompt": prompt,
                "response": response.choices[0].message.content,
                "question_id": idx,
                "prompt_version": prompt_version,
                "model": "gpt-4.1-mini",
                "latency_seconds": latency,
                "input_tokens": response.usage.prompt_tokens,
                "output_tokens": response.usage.completion_tokens,
                "total_tokens": response.usage.total_tokens,
                "input_token_price": input_cost,
                "output_token_price": output_cost,
                "total_cost": total_cost,
                "response_status": "success"


            }
        )

    return pd.DataFrame(results)

In [7]:
df_v1 = evaluate_prompt(
    questions,
    PROMT_V1,
    "v1",
    env['model_name'],
    env['input_token_price'],
    env['output_token_price']
)

print("Statistics of V1 evaluation")
print(df_v1['latency_seconds'].describe())

latency_stats_v1 = df_v1['latency_seconds'].agg(
    ['mean', 'median', "min", 'max', 'std']
).to_frame("value")

print("Selected parameters")
print(latency_stats_v1)

for _, row in df_v1.iterrows():
    print("PROMPT:")
    print(row["prompt"])

    print("\nRESPONSE:")
    print(row["response"])

    print("\n" + "="*80 + "\n")

Statistics of V1 evaluation
count    5.000000
mean     3.399264
std      1.127425
min      2.233298
25%      2.288682
50%      3.451255
75%      4.426579
max      4.596509
Name: latency_seconds, dtype: float64
Selected parameters
           value
mean    3.399264
median  3.451255
min     2.233298
max     4.596509
std     1.127425
PROMPT:

Answer the following student questions clearly.
Question: what is overfitting in machine learning?


RESPONSE:
Overfitting in machine learning occurs when a model learns not only the underlying patterns in the training data but also the noise and outliers. This means that the model performs very well on the training dataset but poorly on unseen or validation data because it has become too complex and specific to the training data rather than capturing the general trends.

Imagine you’re trying to teach a machine to recognize cats in pictures. If the model memorizes all the specific details of the training images, including background objects and light

In [8]:
def add_cost_metrics(df, env):
    """
    Function to calculate and add cost metric to input data frame.
    The method assumes that the dataframe contains input and output token price.
    """
    input_token_price = env['input_token_price']
    output_token_price = env['output_token_price']

    df['input_cost'] = (
        df['input_tokens'] / (1024 * 1024) 
    ) * input_token_price

    df['output_cost'] = (
        df['output_tokens'] / (1024 * 1024)
    ) * output_token_price

    df['total_cost'] = df['input_cost'] + df['output_cost']

    return df

In [9]:
#df_v1 = add_cost_metrics(df=df_v1, env=env)

df_v1.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   prompt              5 non-null      str    
 1   response            5 non-null      str    
 2   question_id         5 non-null      int64  
 3   prompt_version      5 non-null      str    
 4   model               5 non-null      str    
 5   latency_seconds     5 non-null      float64
 6   input_tokens        5 non-null      int64  
 7   output_tokens       5 non-null      int64  
 8   total_tokens        5 non-null      int64  
 9   input_token_price   5 non-null      float64
 10  output_token_price  5 non-null      float64
 11  total_cost          5 non-null      float64
 12  response_status     5 non-null      str    
dtypes: float64(4), int64(4), str(5)
memory usage: 8.4 KB


In [10]:
# Average monthly cost
avg_cost = df_v1['total_cost'].mean()

print(f"Average cost per question: ${avg_cost:.4f}")

projected_monthly_cost = avg_cost * 10000

print(f"Project monthly average cost: ${projected_monthly_cost:.2f}", )

Average cost per question: $0.0002
Project monthly average cost: $1.64


In [11]:
# Promt V2
PROMT_V2 = """
You are an academic assistant.
Answer the following student question in a clear, concise and accurate manner.
Use simple language suitable for undergraduate students.
Question: {Question}
"""

In [12]:
df_v2 = evaluate_prompt(
    questions,
    PROMT_V2,
    "v2",
    env['model_name'],
    env['input_token_price'],
    env['output_token_price']
)

print("Statistics of V2 evaluation")
print(df_v2['latency_seconds'].describe())

latency_stats_v2 = df_v2['latency_seconds'].agg(
    ['mean', 'median', "min", 'max', 'std']
).to_frame("value")

print("Selected parameters")
print(latency_stats_v2)

for _, row in df_v2.iterrows():
    print("PROMPT:")
    print(row["prompt"])

    print("\nRESPONSE:")
    print(row["response"])

    print("\n" + "="*80 + "\n")
# Average monthly cost
avg_cost_v2 = df_v2['total_cost'].mean()

print(f"Average cost per question: ${avg_cost_v2:.4f}")

projected_monthly_cost_v2 = avg_cost_v2 * 10000

print(f"Project monthly average cost: ${projected_monthly_cost_v2:.2f}", )

Statistics of V2 evaluation
count    5.000000
mean     2.863375
std      0.709543
min      1.713696
25%      2.776106
50%      2.950018
75%      3.351021
max      3.526037
Name: latency_seconds, dtype: float64
Selected parameters
           value
mean    2.863375
median  2.950018
min     1.713696
max     3.526037
std     0.709543
PROMPT:

You are an academic assistant.
Answer the following student question in a clear, concise and accurate manner.
Use simple language suitable for undergraduate students.
Question: what is overfitting in machine learning?


RESPONSE:
Overfitting in machine learning occurs when a model learns the training data too well, including its noise and outliers, rather than just the underlying patterns. This means that while the model performs very well on the training data, it does poorly on new, unseen data. 

Think of it like memorizing the answers to a practice test instead of understanding the material. When faced with different questions on the actual exam, a

In [13]:
PROMPT_V3 = """
You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Use this exact JSON schema:
{{
    "answer": "string",
    "confidence": "high|medium|low",
    "key_points": ["string"]
}}

Question" {Question}
"""

In [14]:
df_v3 = evaluate_prompt(
    questions,
    PROMPT_V3,
    "v3",
    env['model_name'],
    env['input_token_price'],
    env['output_token_price']
)

print("Statistics of V3 evaluation")
print(df_v3['latency_seconds'].describe())

latency_stats_v3 = df_v3['latency_seconds'].agg(
    ['mean', 'median', "min", 'max', 'std']
).to_frame("value")

print("Selected parameters")
print(latency_stats_v3)

for _, row in df_v3.iterrows():
    print("PROMPT:")
    print(row["prompt"])

    print("\nRESPONSE:")
    print(row["response"])

    print("\n" + "="*80 + "\n")
# Average monthly cost
avg_cost_v3 = df_v3['total_cost'].mean()

print(f"Average cost per question: ${avg_cost_v3:.4f}")

projected_monthly_cost_v3 = avg_cost_v3 * 10000

print(f"Project monthly average cost: ${projected_monthly_cost_v3:.2f}", )

Statistics of V3 evaluation
count    5.000000
mean     2.108013
std      0.365384
min      1.819644
25%      1.874551
50%      1.987111
75%      2.133415
max      2.725343
Name: latency_seconds, dtype: float64
Selected parameters
           value
mean    2.108013
median  1.987111
min     1.819644
max     2.725343
std     0.365384
PROMPT:

You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Use this exact JSON schema:
{
    "answer": "string",
    "confidence": "high|medium|low",
    "key_points": ["string"]
}

Question" what is overfitting in machine learning?


RESPONSE:
{
    "answer": "Overfitting in machine learning occurs when a model learns not only the underlying pattern in the training data but also the noise and outliers, resulting in a model that performs well on the training set but poorly on new, unseen data.",
    "confidence": "high",
    "key_points": [
        "Overfitting happens when a model is too complex relative to the amount of t

In [15]:
# JSON validation
import json

def is_valid_json(text):
    try:
        json.loads(text)
        return 1
    except:
        return 0


In [16]:
df_v3["valid_json"] = (
    df_v3["response"].apply(is_valid_json)
)

valid_json_rate = df_v3["valid_json"].mean()
print(valid_json_rate)

1.0


In [17]:
# Prompt performance comparison

comparison_df = pd.DataFrame(
    [
        {
            "Prompt Version": "v1",
            "Avg Latency": df_v1["latency_seconds"].mean(),
            "Avg Input Tokens": df_v1["input_tokens"].mean(),
            "Avg Ouput Tokens": df_v1["output_tokens"].mean(),
            "Avg Total Tokens": df_v1["total_tokens"].mean(),
            "Avg Cost": df_v1["total_cost"].mean(),
            "JSON Valid": 0
        },
        {
            "Prompt Version": "v2",
            "Avg Latency": df_v2["latency_seconds"].mean(),
            "Avg Input Tokens": df_v2["input_tokens"].mean(),
            "Avg Ouput Tokens": df_v2["output_tokens"].mean(),
            "Avg Total Tokens": df_v2["total_tokens"].mean(),
            "Avg Cost": df_v2["total_cost"].mean(),
            "JSON Valid": 0
        },
        {
            "Prompt Version": "v3",
            "Avg Latency": df_v3["latency_seconds"].mean(),
            "Avg Input Tokens": df_v3["input_tokens"].mean(),
            "Avg Ouput Tokens": df_v3["output_tokens"].mean(),
            "Avg Total Tokens": df_v3["total_tokens"].mean(),
            "Avg Cost": df_v3["total_cost"].mean(),
            "JSON Valid": df_v3["valid_json"].mean()
        }
    ]
)

comparison_df

,Prompt Version,Avg Latency,Avg Input Tokens,Avg Ouput Tokens,Avg Total Tokens,Avg Cost,JSON Valid
0,v1,3.399264,24.6,281.2,305.8,0.000164,0.0
1,v2,2.863375,45.6,232.2,277.8,0.000139,0.0
2,v3,2.108013,67.6,131.4,199.0,0.000085,1.0


### Comparison of performance for three types of prompting

The comparison of performance for three types of prompting can be summarized as follows:
* **V1**: This version uses a basic prompt structure that provides minimal context. It is effective for straightforward tasks but may struggle with complex queries.
* **V2**: This version incorporates additional context. This instructs Language Model to assume an Academic Assistant's role.
It tends to perform better on tasks that require understanding nuanced instructions or generating more detailed responses.
* **V3**: This version follows **V2** but enhances further in terms of response format. The prompt expects the result in a JSON format, which is particularly useful for structured data extraction and integration with other systems. It is the most effective for tasks that require precise output formatting and data handling.

### Performance Metrics
* **Token Consumption**: **V1** Consumes the highest number of tokens considering the input context is minimal. The language model continues to generate more response and therefore total tokens are higher. **V1** has lesser input tokens in comparison to **V2** and **V3**, howeever the output tokens are higher.
**V3** has the total least number of tokens due its precise context and clear output structure. Following the output structure, language model generates only relevant information. Hence total tokens are lesser in comparison to **V1** and **V2**.
* **Latency**: Latency is low **V3** in comparison to **V1** and **V2**. The structured output format allows for faster processing and integration with other systems, reducing the time taken to parse and utilize the generated data.
* **Cost**: **V3** has the least cost for the given combination of questions and generated answers. The output tokens generated in **V3** are less in comparison to other prompts due to a well defined structure. This overall leads to lesser token price overall.

In [ ]:
import mlflow

PROJECT_PATH = Path.cwd().parent

mlflow.set_tracking_uri(
    f"sqlite:///{PROJECT_PATH}/mlflow.db"
)

mlflow.set_experiment(
    "eduassist_llmops_assignment"
)

2026/11/03 08:38:32 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/03 08:38:33 INFO mlflow.store.db.utils: Updating database tables
2026/11/03 08:39:11 INFO mlflow.tracking.fluent: Experiment with name 'eduassist_llmops_assignment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///d:/Bits-lab/eduassist-llmops-prompt-evaluation/notebooks/mlruns/1', creation_time=1793675351699, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1793675351699, lifecycle_stage='active', name='eduassist_llmops_assignment', tags={}, trace_location=None, workspace='default'>

In [19]:
def log_prompt_metrics(
        prompt_version,
        df,
        valid_json=0
):
    with mlflow.start_run(run_name=prompt_version):
        mlflow.log_param("prompt_version", prompt_version)
        mlflow.log_metric("avg_latency_seconds", df['latency_seconds'].mean())
        mlflow.log_metric("avg_input_tokens", df["input_tokens"].mean())
        mlflow.log_metric("avg_output_tokens", df["output_tokens"].mean())
        mlflow.log_metric("avg_cost_per_request", df["total_cost"].mean())
        mlflow.log_metric("valid_json", valid_json)


In [20]:
log_prompt_metrics(prompt_version="v1", df=df_v1)
log_prompt_metrics(prompt_version="v2", df=df_v2)
log_prompt_metrics(prompt_version="v3", df=df_v3, valid_json=valid_json_rate)